# Data quality audit — Lausanne residential mutations

**Pipeline notebook 2.** Loads the raw master table built in notebook 1, types the
date columns, removes duplicate events, checks the coherence of `CODE_MUTATION`, and
screens the numeric household variables for implausible values. The audited, typed
table is saved for the downstream notebooks (household state reconstruction,
typology, feature engineering, modelling).

Cleaning here stays evidence-based: structural fixes (typing, exact-duplicate
removal) are applied, while anything requiring a modelling decision (which event
counts as a departure, what to do with out-of-range values) is **reported**, not
silently changed.


## 0. Load the master table & column roles

Reads the master file produced by notebook 1 (Parquet if present, otherwise the
compressed-CSV fallback, in which case columns are read back as text). The
column-role lists are re-declared so this notebook is self-contained. Note the
identifier was already renamed `NOREFCH → id_projet` and `NOAVS`/`IDHAB` dropped in
notebook 1.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR  = Path("/mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data")
MASTER_PQ = DATA_DIR / "masterfile_brut.parquet"
MASTER_GZ = DATA_DIR / "masterfile_brut.csv.gz"

if MASTER_PQ.exists():
    clean = pd.read_parquet(MASTER_PQ)
elif MASTER_GZ.exists():
    clean = pd.read_csv(MASTER_GZ, dtype=str, low_memory=False)
else:
    raise FileNotFoundError("No master file found — run notebook 1 first.")

print(f"Loaded master: {len(clean):,} rows x {clean.shape[1]} columns")

# ---- Column roles (new monthly schema) ---------------------------------
DATE_COLS = ["DATE_EFFECTIVE", "MUTATION_DATE", "DATNAIS", "DDECES", "DETATCIVIL",
             "DATEENTCH", "DATARR", "DATDEP", "DATDEM", "XFERESID"]
PROV = ["source_file", "periode", "annee"]

Loaded master: 863,602 rows x 68 columns


## 1. Date typing — raw inventory, convention detection, then parsing

Two successive failures shaped this cell, both instructive. (1) The original
version assumed the Swiss dotted day-first format and coerced everything else:
the notebook-5 diagnostics proved the corpus was being silently day/month-swapped
(zero parsed days > 12 across 136 batches, NaT rates matching the share of month
days > 12, entry timestamps postdating their file with a seasonal pattern —
every December at exactly zero). (2) A first fix detected the convention but
assumed a **dotted** `A.B.YYYY` pattern — and found none: the raw strings do not
use dots (Excel merely *displays* dates in the local convention; the pasted
examples were misleading). The `dayfirst=True` fallback of the original cell
explains the rest: without an explicit format, pandas ≥ 2 infers one single
format from the first element and applies it corpus-wide — `dayfirst` is a hint,
not a contract.

**Root cause, fully resolved by the raw inventory**: the source strings are
**ISO year-first with a fractional second** (`2015-01-08 16:22:21.0`) — the data
was never malformed. The corruption came from a documented pandas trap: when no
format is given, pandas ≥ 2 guesses one from the first element, and with
`dayfirst=True` it returns `%Y-%d-%m` whenever both middle fields of that first
element are ≤ 12 — swapping month and day in the *guessed format* itself (a
UserWarning is emitted, easily missed in a notebook). Applied corpus-wide: every
true day > 12 became an invalid month (NaT, ~60%), every true day ≤ 12 was
silently swapped. All three diagnostic signatures follow from clean ISO input.

This version therefore assumes nothing:

1. **Raw inventory first** — dtype and sample strings per column are printed
   before any parsing, so the evidence is always visible;
2. **Generalized detection** — separator-agnostic (`.`, `/`, `-`), year-first
   recognised (4-digit leading field), day/month order *measured* (in a
   D?M?YYYY pattern the field exceeding 12 is necessarily the day), per-batch
   re-detection if the corpus is mixed;
3. **Explicit-format cascade** built from the detected shape (with/without time,
   seconds, AM/PM variants);
4. **Hard guards** — zero unparsed non-blank strings on the chaining-critical
   columns, parsed day>12 share near ~0.6 (inverted swap detector), no entry
   timestamp postdating its own file's month.

In [2]:
import re

GEN_RX = re.compile(r"^\s*(\d{1,4})([./\-])(\d{1,2})[./\-](\d{1,4})")
TIME_SUFFIXES = [" %H:%M:%S.%f", " %H:%M:%S", " %H:%M",
                 " %I:%M:%S.%f %p", " %I:%M:%S %p", " %I:%M %p", ""]

def profile(raw, label):
    """Inspect raw strings: dtype, samples, separator, field order."""
    if pd.api.types.is_datetime64_any_dtype(raw):
        return {"kind": "already_datetime", "samples": []}
    s = raw.dropna().astype(str).str.strip()
    s = s[s.ne("")]
    samples = s.unique()[:5].tolist()
    if s.empty:
        return {"kind": "all_blank", "samples": []}
    m = s.str.extract(GEN_RX).dropna()
    if len(m) / len(s) < 0.5:
        return {"kind": "unrecognised", "samples": samples}
    a, sep, b, c = m[0].astype(int), m[1].mode().iloc[0], m[2].astype(int), m[3].astype(int)
    if (a > 31).mean() > 0.5:
        return {"kind": "yearfirst", "sep": sep, "samples": samples}
    a_gt12, b_gt12 = (a > 12).mean(), (b > 12).mean()
    if a_gt12 > 0.10 and b_gt12 < 0.001:
        return {"kind": "dayfirst", "sep": sep, "samples": samples}
    if b_gt12 > 0.10 and a_gt12 < 0.001:
        return {"kind": "monthfirst", "sep": sep, "samples": samples}
    return {"kind": "ambiguous", "sep": sep, "samples": samples,
            "a_gt12": round(a_gt12, 3), "b_gt12": round(b_gt12, 3)}

def date_fmt(kind, sep):
    return {"dayfirst":   f"%d{sep}%m{sep}%Y",
            "monthfirst": f"%m{sep}%d{sep}%Y",
            "yearfirst":  f"%Y{sep}%m{sep}%d"}[kind]

def parse_with(raw, kind, sep):
    # Accumulator in MICROsecond resolution: datetime64[ns] tops out at year 2262,
    # and the register contains millennium typos (e.g. year 3021) that recent
    # pandas returns in a wider unit -- assigning that into [ns] crashes.
    parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[us]")
    todo = raw.notna() & raw.astype(str).str.strip().ne("")
    for suf in TIME_SUFFIXES:
        if not todo.any():
            break
        p = pd.to_datetime(raw[todo], format=date_fmt(kind, sep) + suf, errors="coerce")
        try:
            p = p.dt.as_unit("us")          # harmonise unit (exact: data stops at seconds)
        except AttributeError:
            pass                            # older pandas: unit handling not needed
        parsed.loc[todo] = p
        todo = todo & parsed.isna()
    return parsed

# ---- 1) Raw inventory (printed before any parsing: the evidence stays visible)
profiles = {}
print("=== Raw date-string inventory ===")
for c in [col for col in DATE_COLS if col in clean.columns]:
    profiles[c] = profile(clean[c], c)
    print(f"  {c:15} dtype={str(clean[c].dtype):10} kind={profiles[c]['kind']:16} "
          f"samples={profiles[c]['samples'][:3]}")

# ---- 2) Detection + parsing ------------------------------------------------
report = []
for c, prof in profiles.items():
    raw, kind = clean[c], prof["kind"]
    n_nonblank = (0 if kind == "already_datetime"
                  else int((raw.notna() & raw.astype(str).str.strip().ne("")).sum()))

    if kind == "already_datetime":
        report.append({"column": c, "convention": kind, "non_blank": int(raw.notna().sum()),
                       "parsed": int(raw.notna().sum()), "still_unparsed": 0,
                       "min": raw.min(), "max": raw.max()})
        continue
    if kind in ("all_blank", "unrecognised"):
        report.append({"column": c, "convention": kind, "non_blank": n_nonblank,
                       "parsed": 0, "still_unparsed": n_nonblank, "min": pd.NaT, "max": pd.NaT})
        continue

    if kind == "ambiguous":
        # Mixed conventions across batches: detect and parse per source file.
        parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
        seen = {}
        for per, idx in clean.groupby("periode").groups.items():
            bp = profile(raw.loc[idx], f"{c}@{per}")
            assert bp["kind"] in ("dayfirst", "monthfirst", "yearfirst"), \
                f"{c} @ {per}: still ambiguous at batch level -- {bp}"
            seen[bp["kind"]] = seen.get(bp["kind"], 0) + 1
            parsed.loc[idx] = parse_with(raw.loc[idx], bp["kind"], bp["sep"])
        conv = f"mixed {seen}"
    else:
        parsed = parse_with(raw, kind, prof["sep"])
        conv = f"{kind} ('{prof['sep']}')"

    clean[c] = parsed
    report.append({"column": c, "convention": conv, "non_blank": n_nonblank,
                   "parsed": int(parsed.notna().sum()),
                   "still_unparsed": n_nonblank - int(parsed.notna().sum()),
                   "min": parsed.min(), "max": parsed.max()})

rep = pd.DataFrame(report)
print("\n=== Parse report ===")
print(rep.to_string(index=False))

# ---- Year-plausibility report (typos like year 3021 are reported, not erased)
print("\n=== Implausible years (outside 1875-2100) ===")
any_implausible = False
for c in [col for col in DATE_COLS if col in clean.columns
          and pd.api.types.is_datetime64_any_dtype(clean[col])]:
    yr = clean[c].dt.year
    m = yr.notna() & ((yr < 1875) | (yr > 2100))
    if m.any():
        any_implausible = True
        vals = clean.loc[m, c].dt.date.astype(str).unique()[:5].tolist()
        print(f"  {c:15} {int(m.sum()):>5,} rows  samples={vals}")
if not any_implausible:
    print("  none")

# ---- 3) Hard guards ---------------------------------------------------------
for c in ["DATE_EFFECTIVE", "MUTATION_DATE"]:
    still = int(rep.loc[rep["column"] == c, "still_unparsed"].iloc[0])
    assert still == 0, f"{c}: {still:,} non-blank strings remain unparsed -- see inventory samples above"

for c in ["DATE_EFFECTIVE", "MUTATION_DATE"]:
    share = float((clean[c].dt.day > 12).sum() / max(int(clean[c].notna().sum()), 1))
    assert 0.40 < share < 0.80, f"{c}: day>12 share {share:.2f} -- swap signature still present?"
    print(f"  {c}: day>12 share among parsed = {share:.2f} (expected ~0.6) OK")

per_end = pd.PeriodIndex(clean["periode"], freq="M").end_time
n_future = int((clean["MUTATION_DATE"] > per_end).sum())
print(f"  MUTATION_DATE entries postdating their file month: {n_future:,} "
      + ("OK" if n_future == 0 else "<- inspect (should be ~0 after the fix)"))

=== Raw date-string inventory ===
  DATE_EFFECTIVE  dtype=object     kind=yearfirst        samples=['2015-01-01 00:00:00.0', '2015-01-05 00:00:00.0', '2014-11-12 00:00:00.0']
  MUTATION_DATE   dtype=object     kind=yearfirst        samples=['2015-01-08 16:22:21.0', '2015-01-05 12:59:53.0', '2015-01-05 12:59:29.0']
  DATNAIS         dtype=object     kind=yearfirst        samples=['1981-04-10 00:00:00.0', '1959-12-04 00:00:00.0', '1979-09-19 00:00:00.0']
  DDECES          dtype=object     kind=yearfirst        samples=['2014-12-31 00:00:00.0', '2015-01-05 00:00:00.0', '2015-01-08 00:00:00.0']
  DETATCIVIL      dtype=object     kind=yearfirst        samples=['1981-04-10 00:00:00.0', '2005-11-29 00:00:00.0', '2013-03-01 00:00:00.0']
  DATEENTCH       dtype=object     kind=yearfirst        samples=['1982-12-23 00:00:00.0', '2015-01-06 00:00:00.0', '2014-02-20 00:00:00.0']
  DATARR          dtype=object     kind=yearfirst        samples=['2014-03-20 00:00:00.0', '1983-02-01 00:00:00.0', '200

## 2. Duplicate-event audit

Monthly exports can repeat an event if extraction windows overlap or a month was
re-imported. Two notions are distinguished: *exact* duplicates (identical on every
substantive column — pure repeats, safe to drop) and *event-key* duplicates (same
individual, same dates, same mutation type, but a differing payload — possibly a
genuine second event, kept for review). Exact duplicates are removed (keeping the
first occurrence); key conflicts are only reported.


In [3]:
substantive = [c for c in clean.columns if c not in PROV]

# 1) Exact duplicates on substantive columns
exact_dup = clean.duplicated(subset=substantive, keep="first")
print(f"Exact duplicate rows: {exact_dup.sum():,} ({exact_dup.mean()*100:.2f}%)")

# 2) Event-key duplicates
event_key = [c for c in ["id_projet", "MUTATION_DATE", "DATE_EFFECTIVE", "CODE_MUTATION"]
             if c in clean.columns]
key_dup      = clean.duplicated(subset=event_key, keep=False)
key_conflict = key_dup & ~clean.duplicated(subset=substantive, keep=False)
print(f"Rows sharing an event key {event_key}: {key_dup.sum():,}")
print(f"  -> same key but different payload (genuine multi-events?): {key_conflict.sum():,}")

# Remove exact duplicates only (decision-free); keep key conflicts for inspection.
before = len(clean)
clean = clean.drop_duplicates(subset=substantive, keep="first").reset_index(drop=True)
print(f"\nRemoved {before - len(clean):,} exact duplicates -> {len(clean):,} rows")

Exact duplicate rows: 25 (0.00%)
Rows sharing an event key ['id_projet', 'MUTATION_DATE', 'DATE_EFFECTIVE', 'CODE_MUTATION']: 84
  -> same key but different payload (genuine multi-events?): 34

Removed 25 exact duplicates -> 863,577 rows


## 3. Mutation-code coherence

`CODE_MUTATION` defines the event type and underpins the binary departure target
built later, so its distribution, stability over time, and consistency with
`TYPE_ADRESSE` / `DATDEP` are checked here. Departure-like codes are identified by
name and cross-validated against the presence of a departure date — a departure code
without `DATDEP` (or the reverse) signals an inconsistency to document.


In [4]:
vc = clean["CODE_MUTATION"].value_counts(dropna=False)
print("CODE_MUTATION distribution:")
print(vc.to_string())
print(f"\nMissing CODE_MUTATION: {clean['CODE_MUTATION'].isna().sum():,}")

# Departure-like codes (feed the binary target)
dep_codes = [v for v in vc.index if isinstance(v, str) and "depart" in v.lower()]
print("\nDeparture-like codes:", dep_codes)

if "TYPE_ADRESSE" in clean.columns:
    print("\nTYPE_ADRESSE distribution:")
    print(clean["TYPE_ADRESSE"].value_counts(dropna=False).to_string())

is_dep = clean["CODE_MUTATION"].isin(dep_codes)
print(f"\nRows with a departure code: {is_dep.sum():,}")
if "DATDEP" in clean.columns:
    print(f"  with DATDEP present : {clean.loc[is_dep, 'DATDEP'].notna().mean()*100:.1f}%")
    odd = clean["DATDEP"].notna() & ~is_dep
    print(f"  DATDEP present but non-departure code: {odd.sum():,}")

# Stability of the main codes over time
top = vc.head(8).index
pivot = (clean[clean["CODE_MUTATION"].isin(top)]
         .pivot_table(index="annee", columns="CODE_MUTATION",
                      values="id_projet", aggfunc="count", fill_value=0))
print("\nTop CODE_MUTATION counts by year:")
print(pivot.to_string())

CODE_MUTATION distribution:
CODE_MUTATION
MutationDepartDefinitif                  199440
MutationDemenagement                     141751
MutationArriveeProvisoireEnDefinitive    127358
MutationDepartAnticipe                    74206
MutationArriveeDefinitive                 68795
CorrectionAdresseEffective                60938
CorrectionRegroupementMenage              54817
MutationDepartNonConfirme                 43662
CorrectionSejour                          20956
MutationNaissance                         18440
MutationDeces                             11865
SuppressionDepartDefinitif                 9767
SuppressionArriveeProvisoire               9704
MutationRegroupementMenage                 4735
MutationAdresseAdministrative              4075
SuppressionDemenagement                    2714
SuppressionDepartNonConfirme               2054
SuppressionDepartAnticipe                  1627
SuppressionDossier                         1408
CorrectionDegroupementMenage               134

## 4. Numeric plausibility (household variables)

Household size (`PERSMEN`), room count (`NOMBRE_PIECE`, half-rooms allowed) and
dwelling surface (`SURFACE`), plus their before-mutation counterparts, are converted
to numbers — handling the Swiss decimal comma — and screened for implausible values.
Non-null strings that fail numeric parsing are counted separately from genuine
missings. Bounds are diagnostic thresholds for discussion, not automatic
corrections. `NaN` on these columns is expected for events that do not concern
housing (e.g. a marital-status change), reflecting the event-based nature of the data.


In [5]:
num_cols = [c for c in ["PERSMEN", "NOMBRE_PIECE", "SURFACE",
                        "MUTATION_PERSMEN", "MUTATION_PIECES", "MUTATION_SURFACE"]
            if c in clean.columns]

def to_numeric_swiss(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False).str.strip(),
                         errors="coerce")

for c in num_cols:
    before_nn = clean[c].notna().sum()
    clean[c]  = to_numeric_swiss(clean[c])
    lost = before_nn - clean[c].notna().sum()
    if lost:
        print(f"  {c}: {lost:,} non-null values failed numeric parse")

print("\nDescriptive statistics:")
print(clean[num_cols].describe().T.to_string())

bounds = {"PERSMEN": (1, 30), "MUTATION_PERSMEN": (1, 30),
          "NOMBRE_PIECE": (0.5, 20), "MUTATION_PIECES": (0.5, 20),
          "SURFACE": (5, 1000), "MUTATION_SURFACE": (5, 1000)}
print("\nImplausible values (out of bounds):")
for c in num_cols:
    lo, hi = bounds[c]
    v = clean[c]
    print(f"  {c:18} nan={v.isna().sum():>7,}  < {lo}: {int((v < lo).sum()):>6,}"
          f"  > {hi}: {int((v > hi).sum()):>6,}")


Descriptive statistics:
                     count       mean        std  min   25%   50%   75%    max
PERSMEN           863577.0   1.943627   1.457328  0.0   1.0   2.0   3.0   14.0
NOMBRE_PIECE      740019.0   2.914044   1.391289  1.0   2.0   3.0   4.0   20.0
SURFACE           740006.0  74.502411  40.091401  7.0  50.0  70.0  92.0  980.0
MUTATION_PERSMEN  863577.0   1.499651   1.414383  0.0   0.0   1.0   2.0   13.0
MUTATION_PIECES   571577.0   2.863242   1.393924  1.0   2.0   3.0   4.0   20.0
MUTATION_SURFACE  571568.0  73.080291  40.062882  7.0  49.0  68.0  90.0  980.0

Implausible values (out of bounds):
  PERSMEN            nan=      0  < 1: 111,691  > 30:      0
  NOMBRE_PIECE       nan=123,558  < 0.5:      0  > 20:      0
  SURFACE            nan=123,571  < 5:      0  > 1000:      0
  MUTATION_PERSMEN   nan=      0  < 1: 231,940  > 30:      0
  MUTATION_PIECES    nan=292,000  < 0.5:      0  > 20:      0
  MUTATION_SURFACE   nan=292,009  < 5:      0  > 1000:      0


## 5. Key & coherence validation

A final set of integrity and descriptive checks before the audited table feeds the
trajectory work. Nothing is modified here — every check is reported.

**Part A — integrity (must hold before Phase 2/3):** the individual key `id_projet`
and household key `NUMERO_MENAGE`, the arrival anchor `DATARR` (which resolves the
left-truncation of presence duration), and the logical ordering of personal dates.
Events dated before the declared arrival are expected in small numbers — `DATARR` is
the *last* arrival, so return migration legitimately produces earlier events.

**Part B — completeness & descriptive profiles (for the report's data section):**
month-level completeness of the corpus, the per-column missingness profile, and the
distributions of the key categorical variables (including the four-way `TYPE_MENAGE`
and the `CODETA_C` lifecycle state). Note: much of the missingness is *structural* —
`COMPROV`/`COMNAIS` are only filled when provenance/birth country is Switzerland
(`PAYSPROVN`/`PAYSNAIS` = 8100), and `MUTATION_*` dwelling fields are empty on
arrivals and departures. These are informative, not defects, and must not be imputed.


In [6]:
# ---- Part A: integrity checks --------------------------------------------
print("Keys")
print(f"  id_projet     : null={clean['id_projet'].isna().sum():,}  "
      f"unique individuals={clean['id_projet'].nunique():,}")
ev_per_ind = clean.groupby("id_projet").size()
print(f"  events/individual: mean={ev_per_ind.mean():.1f}  "
      f"median={ev_per_ind.median():.0f}  max={ev_per_ind.max():,}")
if "NUMERO_MENAGE" in clean.columns:
    print(f"  NUMERO_MENAGE : null={clean['NUMERO_MENAGE'].isna().mean()*100:.1f}%  "
          f"unique households={clean['NUMERO_MENAGE'].nunique():,}")

print("\nDATARR (arrival anchor)")
print(f"  missing: {clean['DATARR'].isna().mean()*100:.1f}%")
if {"DATARR", "DATE_EFFECTIVE"} <= set(clean.columns):
    both       = clean["DATARR"].notna() & clean["DATE_EFFECTIVE"].notna()
    before_arr = (clean["DATE_EFFECTIVE"] < clean["DATARR"]) & both
    print(f"  events dated before declared arrival: {int(before_arr.sum()):,} "
          f"({before_arr.sum()/max(int(both.sum()),1)*100:.2f}% of datable rows) "
          f"-- expected for return migration")

def violations(a, b, label):
    if a in clean.columns and b in clean.columns:
        m   = clean[a].notna() & clean[b].notna()
        bad = clean.loc[m, a] > clean.loc[m, b]
        print(f"  {label}: {int(bad.sum()):,} violations / {int(m.sum()):,} comparable")

print("\nCross-date coherence (a should not be after b)")
violations("DATNAIS", "DATARR", "DATNAIS > DATARR")
violations("DATARR",  "DATDEP", "DATARR  > DATDEP")
violations("DATNAIS", "DDECES", "DATNAIS > DDECES")

Keys
  id_projet     : null=0  unique individuals=291,763
  events/individual: mean=3.0  median=2  max=39
  NUMERO_MENAGE : null=0.0%  unique households=194,061

DATARR (arrival anchor)
  missing: 0.0%
  events dated before declared arrival: 7,047 (0.82% of datable rows) -- expected for return migration

Cross-date coherence (a should not be after b)
  DATNAIS > DATARR: 4 violations / 863,569 comparable
  DATARR  > DATDEP: 21 violations / 334,712 comparable
  DATNAIS > DDECES: 0 violations / 12,352 comparable


In [7]:
# ---- Part B: completeness & descriptive profiles -------------------------
found    = sorted(clean["periode"].unique())
expected = pd.period_range(found[0], found[-1], freq="M").astype(str).tolist()
missing_months = sorted(set(expected) - set(found))
print(f"Months: {len(found)} found / {len(expected)} expected "
      f"({found[0]} .. {found[-1]})")
print("  missing months:", missing_months or "none")

miss = (clean.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nMissing rate by column (%, top 20):")
print(miss.head(20).to_string())
print(f"  fully empty: {(miss == 100).sum()}   |   complete: {(miss == 0).sum()}")

for c in ["CODE_SEXE", "ETAT_CIVIL", "TYPE_MENAGE", "TYPADRES_C", "CODETA_C"]:
    if c in clean.columns:
        print(f"\n{c}:")
        print(clean[c].value_counts(dropna=False).to_string())
if "NATION" in clean.columns:
    print(f"\nNATION: {clean['NATION'].nunique():,} distinct values, top 10:")
    print(clean["NATION"].value_counts(dropna=False).head(10).to_string())

Months: 137 found / 137 expected (2015-01 .. 2026-05)
  missing months: none

Missing rate by column (%, top 20):
DDECES                98.6
XFERESID              98.4
COMDEST               93.3
COMDEST_C             93.3
AUTRECOM              92.9
AUTRPAYS              92.4
COMNAIS               64.9
DATDEP                61.2
PAYSDEST              61.2
PAYSDEST_C            61.2
TYPE_ADRESSE          61.2
MUTATION_COMPROV_C    56.3
MUTATION_COMPROV      56.3
COMPROV               43.1
COMPROV_C             43.1
DATEENTCH             36.1
PERMIS_C              34.9
TYPERM_C              34.9
MUTATION_SURFACE      33.8
MUTATION_PIECES       33.8
  fully empty: 0   |   complete: 22

CODE_SEXE:
CODE_SEXE
Masculin    445256
Feminin     418282
Inconnu         39

ETAT_CIVIL:
ETAT_CIVIL
Celibataire                    623909
Marie                          155957
Divorce                         43349
SepareDeFait                    18421
Veuf                            13830
SepareLegal      

## 6. Save the audited master & next steps

The typed, de-duplicated table is saved as `masterfile_audited` (Parquet, with a
compressed-CSV fallback). **Next notebook:** reconstruct household state at a
reference date using `NUMERO_MENAGE` and the `MUTATION_*` before-state, then build
the household typology (single-person / couple ± children / single-parent /
non-family / collective), inferring composition from `ETAT_CIVIL` + `DATNAIS` within
each household.


In [8]:
OUT_PQ = DATA_DIR / "masterfile_audited.parquet"
try:
    clean.to_parquet(OUT_PQ, index=False)
    saved = OUT_PQ
except Exception as e:
    saved = OUT_PQ.with_suffix(".csv.gz")
    clean.to_csv(saved, index=False, compression="gzip")
    print(f"Parquet unavailable ({type(e).__name__}) -> CSV gzip.")
print(f"Saved: {saved} ({saved.stat().st_size/1e6:.1f} MB)")

Saved: /mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data/masterfile_audited.parquet (58.3 MB)
